# Image Dataset Cleaning — PyTorch
### MedAI Diagnose | CNN + NLP + PEPA
**Run order:** Step 2 — after clean_text_datasets.ipynb

**What this does:**
- Finds images in all 10 Kaggle dataset folders
- Resizes every image to 224×224 RGB
- Maps folder names → disease label names
- Saves organised to `dataset/cleaned/images/<disease>/`
- Compatible with PyTorch `torchvision.transforms`

**Output:** `dataset/cleaned/images/<disease_name>/00001.jpg`

In [2]:
# Cell 1 — Imports and Config
import os
import glob
import warnings
import numpy as np
from PIL import Image
from pathlib import Path
warnings.filterwarnings('ignore')

# ── ABSOLUTE PATHS — works regardless of where VS Code runs notebook from ──
BASE     = r'C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose'
RAW_BASE = BASE + r'\dataset\raw\images'
OUT_BASE = BASE + r'\dataset\cleaned\images'

PATHS = {
    'covid_chest':  RAW_BASE + r'\covid19-radiography-database',
    'nih_chest':    RAW_BASE + r'\nih-chest-xrays',
    'chest_ct':     RAW_BASE + r'\chest-ctscan-images',
    'brain_tumor':  RAW_BASE + r'\brain-tumor-mri-dataset',
    'alzheimer':    RAW_BASE + r'\augmented-alzheimer-mri-dataset',
    'brain_stroke': RAW_BASE + r'\brain-stroke-ct-image-dataset',
    'isic_skin':    RAW_BASE + r'\isic-2019',
    'ham10000':     RAW_BASE + r'\skin-cancer-mnist-ham10000',
    'dermnet':      RAW_BASE + r'\dermnet',
    'skin_disease': RAW_BASE + r'\skin-disease-dataset',
    'eye_disease':  RAW_BASE + r'\eye-diseases-classification',
    'retinal_oct':  RAW_BASE + r'\kermany2018',
    'ecg':          RAW_BASE + r'\heartbeat',
    'thyroid':      RAW_BASE + r'\ddti-thyroid-ultrasound-images',
    'knee':         RAW_BASE + r'\knee-osteoarthritis-dataset-with-severity',
    'mura_bone':    RAW_BASE + r'\mura-v11',
    'liver_ct':     RAW_BASE + r'\liver-tumor-segmentation',
    'kidney_ct':    RAW_BASE + r'\ct-kidney-dataset-normal-cyst-tumor-and-stone',
    'malaria':      RAW_BASE + r'\cell-images-for-detecting-malaria',
}

# ── Image config ─────────────────────────────────────────────────
IMG_SIZE      = (224, 224)
IMG_EXTS      = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
MAX_PER_CLASS = 500
MIN_PER_CLASS = 20

os.makedirs(OUT_BASE, exist_ok=True)

# ── Verify which datasets are present ────────────────────────────
print(f'BASE     : {BASE}')
print(f'RAW_BASE : {RAW_BASE}')
print(f'OUT_BASE : {OUT_BASE}')
print()
print('Dataset folder check:')
for key, path in PATHS.items():
    status = '✓ FOUND' if os.path.isdir(path) else '✗ MISSING'
    print(f'  {key:<20} {status}   {path}')

BASE     : C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose
RAW_BASE : C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images
OUT_BASE : C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\cleaned\images

Dataset folder check:
  covid_chest          ✓ FOUND   C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\covid19-radiography-database
  nih_chest            ✓ FOUND   C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\nih-chest-xrays
  chest_ct             ✓ FOUND   C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\chest-ctscan-images
  brain_tumor          ✓ FOUND   C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\brain-tumor-mri-dataset
  alzheimer            ✓ FOUND   C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\augmented-alzheimer-mri-dataset
  brain_stroke         ✓ FOUND   C:\Users\Aditya 

In [3]:
# Cell 2 — Label Maps (Kaggle folder names → disease labels)

LABEL_MAPS = {
    'covid_chest':  {'covid':'covid','lung_opacity':'pneumonia','viral pneumonia':'pneumonia','normal':'normal chest','pneumonia':'pneumonia'},
    'nih_chest':    {'atelectasis':'atelectasis','cardiomegaly':'cardiomegaly','effusion':'pleural effusion','pneumonia':'pneumonia','edema':'pulmonary edema','emphysema':'emphysema','fibrosis':'pulmonary fibrosis','pneumothorax':'pneumothorax','no finding':'normal chest'},
    'chest_ct':     {'adenocarcinoma':'lung cancer','large.cell.carcinoma':'lung cancer','squamous.cell.carcinoma':'lung cancer','normal':'normal chest','n':'normal chest'},
    'brain_tumor':  {'glioma':'brain cancer','meningioma':'meningioma','pituitary':'pituitary tumor','notumor':'normal brain','no_tumor':'normal brain','normal':'normal brain'},
    'alzheimer':    {'nondemented':'normal brain','verymilddemented':'alzheimer disease','milddemented':'alzheimer disease','moderatedemented':'alzheimer disease','non_demented':'normal brain','mild':'alzheimer disease','moderate':'alzheimer disease'},
    'brain_stroke': {'stroke':'stroke','normal':'normal brain','positive':'stroke','negative':'normal brain'},
    'isic_skin':    {'mel':'melanoma','melanoma':'melanoma','nv':'skin disorder','bcc':'skin cancer','bkl':'seborrheic keratosis','df':'dermatofibroma','vasc':'vascular lesion','akiec':'actinic keratosis','scc':'skin cancer'},
    'ham10000':     {'mel':'melanoma','melanoma':'melanoma','nv':'skin disorder','bcc':'skin cancer','bkl':'seborrheic keratosis','df':'dermatofibroma','vasc':'vascular lesion','akiec':'actinic keratosis'},
    'dermnet':      {'eczema':'eczema','psoriasis':'psoriasis','acne':'acne','rosacea':'rosacea','ringworm':'fungal infection of the skin','chickenpox':'chickenpox','shingles':'shingles herpes zoster','nail fungus':'fungal infection','seborrheic dermatitis':'seborrheic dermatitis'},
    'skin_disease': {'eczema':'eczema','psoriasis':'psoriasis','acne':'acne','melanoma':'melanoma','normal':'normal skin','fungal':'fungal infection of the skin'},
    'eye_disease':  {'diabetic_retinopathy':'diabetic retinopathy','cataract':'cataract','glaucoma':'glaucoma','normal':'normal eye'},
    'retinal_oct':  {'cnv':'diabetic retinopathy','dme':'diabetic retinopathy','drusen':'glaucoma','normal':'normal eye'},
    'ecg':          {'normal':'normal cardiac','sveb':'arrhythmia','veb':'arrhythmia','0':'normal cardiac','1':'arrhythmia','2':'arrhythmia','3':'arrhythmia','4':'arrhythmia'},
    'thyroid':      {'benign':'thyroid disease','malignant':'thyroid cancer','normal':'normal thyroid'},
    'knee':         {'0':'normal knee','1':'osteoarthritis','2':'osteoarthritis','3':'osteoarthritis','4':'osteoarthritis','normal':'normal knee'},
    'mura_bone':    {'positive':'bone fracture','negative':'normal bone','1':'bone fracture','0':'normal bone','abnormal':'bone fracture','normal':'normal bone'},
    'liver_ct':     {'tumor':'liver cancer','cancer':'liver cancer','normal':'normal liver','cyst':'liver cyst'},
    'kidney_ct':    {'cyst':'benign kidney cyst','normal':'normal kidney','stone':'kidney stone','tumor':'kidney cancer'},
    'malaria':      {'parasitized':'malaria','infected':'malaria','uninfected':'normal blood','healthy':'normal blood'},
}
print(f'✓ Label maps loaded for {len(LABEL_MAPS)} datasets')

✓ Label maps loaded for 19 datasets


In [4]:
# Cell 3 — Processing Functions

def find_images(folder):
    imgs = []
    for ext in IMG_EXTS:
        imgs += glob.glob(os.path.join(folder, '**', f'*{ext}'), recursive=True)
    return imgs

def save_image(src, dst):
    """Resize to 224x224 RGB and save as JPEG."""
    try:
        img = Image.open(src).convert('RGB').resize(IMG_SIZE, Image.BILINEAR)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        img.save(dst, 'JPEG', quality=90)
        return True
    except:
        return False

def match_label(folder_name, label_map):
    fn = folder_name.lower().strip()
    for kw, label in label_map.items():
        if kw.lower() in fn or fn in kw.lower():
            return label
    return None

def process_dataset(key, raw_path, label_map):
    if not os.path.isdir(raw_path):
        print(f'  [SKIP] Not found: {raw_path}')
        return {}

    print(f'\n  {key}  ({raw_path})')

    # Collect all class folders
    class_folders = []
    for entry in os.listdir(raw_path):
        full = os.path.join(raw_path, entry)
        if os.path.isdir(full): class_folders.append((entry, full))

    if not class_folders:
        for sub in os.listdir(raw_path):
            sf = os.path.join(raw_path, sub)
            if os.path.isdir(sf):
                for entry in os.listdir(sf):
                    full = os.path.join(sf, entry)
                    if os.path.isdir(full): class_folders.append((entry, full))

    for split in ['train', 'Train', 'training', 'Training']:
        sp = os.path.join(raw_path, split)
        if os.path.isdir(sp):
            for entry in os.listdir(sp):
                full = os.path.join(sp, entry)
                if os.path.isdir(full): class_folders.append((entry, full))

    counts = {}
    for fname, fpath in class_folders:
        label = match_label(fname, label_map) or fname.lower().replace('_', ' ')
        if not label: continue

        safe   = label.replace(' ', '_').replace('/', '_')
        out_d  = os.path.join(OUT_BASE, safe)
        exist  = len(glob.glob(os.path.join(out_d, '*.jpg')))
        remain = MAX_PER_CLASS - exist
        if remain <= 0: continue

        imgs = find_images(fpath)
        np.random.seed(42)
        np.random.shuffle(imgs)

        saved = 0
        for i, src in enumerate(imgs[:remain]):
            dst = os.path.join(out_d, f'{exist+i:05d}.jpg')
            if save_image(src, dst): saved += 1

        counts[label] = counts.get(label, 0) + saved
        print(f'    {label:<45} +{saved:>4}  (total: {exist+saved})')

    return counts

print('✓ Processing functions ready')

✓ Processing functions ready


In [5]:
# Cell 4 — Process All Datasets
print('='*55)
print('  PROCESSING ALL IMAGE DATASETS')
print('='*55)

all_counts = {}
for key, raw_path in PATHS.items():
    counts = process_dataset(key, raw_path, LABEL_MAPS.get(key, {}))
    for label, count in counts.items():
        all_counts[label] = all_counts.get(label, 0) + count

print('\n✓ All datasets processed')

  PROCESSING ALL IMAGE DATASETS

  covid_chest  (C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\covid19-radiography-database)
    covid                                         + 500  (total: 500)

  nih_chest  (C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\raw\images\nih-chest-xrays)
    images 001                                    + 500  (total: 500)
    images 002                                    + 500  (total: 500)
    images 003                                    + 500  (total: 500)
    images 004                                    + 500  (total: 500)
    images 005                                    + 500  (total: 500)
    images 006                                    + 500  (total: 500)
    images 007                                    + 500  (total: 500)
    images 008                                    + 500  (total: 500)
    images 009                                    + 500  (total: 500)
    images 010                   

In [6]:
# Cell 5 — Summary
print('='*55)
print('  FINAL SUMMARY')
print('='*55)
print(f'  Output folder : {OUT_BASE}')
print(f'  Total classes : {len(all_counts)}')
print(f'  Total images  : {sum(all_counts.values()):,}')
print(f'  Image size    : {IMG_SIZE[0]}x{IMG_SIZE[1]} RGB (ready for EfficientNetB0)')

ready = {k:v for k,v in sorted(all_counts.items(), key=lambda x:-x[1]) if v >= MIN_PER_CLASS}
print(f'\n  Classes ready for CNN training ({MIN_PER_CLASS}+ images):')
for label, count in ready.items():
    bar = '█' * min(30, count // 10)
    print(f'    {label:<45} {count:>5}  {bar}')

print(f'\n✓ Image cleaning complete')
print(f'✓ {len(ready)} disease classes ready for PyTorch training')

  FINAL SUMMARY
  Output folder : C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\dataset\cleaned\images
  Total classes : 67
  Total images  : 29,902
  Image size    : 224x224 RGB (ready for EfficientNetB0)

  Classes ready for CNN training (20+ images):
    covid                                           500  ██████████████████████████████
    images 001                                      500  ██████████████████████████████
    images 002                                      500  ██████████████████████████████
    images 003                                      500  ██████████████████████████████
    images 004                                      500  ██████████████████████████████
    images 005                                      500  ██████████████████████████████
    images 006                                      500  ██████████████████████████████
    images 007                                      500  ██████████████████████████████
    images 008               